# Семинар №4. Оптимизация

**План**
1. Попробуем написать логистическую регрессию руками

## Импорты

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.datasets import load_iris, make_classification
from sklearn.metrics import classification_report

## Logistic regression

Из тетрадки по логистической регрессии:
$y(x_1, ..., x_n) = \sigma (w_0 + w_1 * x_1 + ... + w_n * x_n) = \sigma (\langle w, x \rangle + w_0)$,

где
- $w$ - вектор весов $w_1, ..., w_n$
- $x$ - вектор признаков $x_1, ..., x_n$
- $\langle w, x \rangle$ - скалярное произведение
- $\sigma(x) = \frac{1}{1 + e^{-x}}$ - логистическая функция (или сигмоида, хотя сигмоида на самом деле - это класс функций)

_Напомню, что по определению скалярного произведения $\langle a, b \rangle = \displaystyle\sum_{i=1}^n a_i * b_i$_

Для простоты будем считать, что у нас:

$y(x_1, ..., x_n) = \sigma (\langle w, x \rangle)$


Сделаем случайные данные для классификации, сначала для двух признаков, чтобы было можно рисовать результат

In [ ]:
n_features = 2

X, y = make_classification(n_samples=100, n_features=n_features, n_informative=2,
                           n_redundant=0, n_repeated=0, n_classes=2,
                           shuffle=True, random_state=42)
X.shape, y.shape

Можно ли с нашими данными работать без свободного коэффициента регрессии?

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ax.set_xlim(-3, 3), ax.set_ylim(-3, 3)

ax.scatter(X[y == 0, 0], X[y == 0, 1], label='Class 0')
ax.scatter(X[y == 1, 0], X[y == 1, 1], label='Class 1')
ax.plot([-3, 3], [-6, 6], label='some line', color='red')
ax.legend();

В нашей регресси весов должно быть столько же, сколько и признаков (так как свободный коэффициент мы решили не брать). Сделаем вектор с ними, пока случайный

In [ ]:
W = np.random.rand(n_features)
W

Теперь поймем, как из этого посчитать предсказание логистической регрессии

$y(x_1, ..., x_n) = \sigma (\langle w, x \rangle)$

In [ ]:
pred = ...
pred

In [ ]:
def predict(W, X, eps=1e-15, threshold=None):
    pred = ...

    if threshold is not None:
        pred = ...

    return pred

Теперь нам надо выбрать какую-то функцию потерь. Пусть это будет кросс-энтропия

$\mathrm{BCE} = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \cdot \log(\hat{y_i}) + (1 - y_i) \cdot \log(1 - \hat{y_i}) \right]$

In [ ]:
eps = 1e-15

# чтобы не получить логарифм от нуля
clip_pred = np.clip(pred, eps, 1 - eps)

# сама кросс-энтропия
loss = ...

np.mean(loss)

In [ ]:
def compute_loss(pred, y, eps=1e-15):
    loss = ...
    return np.mean(loss)

А теперь нам надо понять, как запустить градиентный спуск для этого результата.

- Что у нас будет переменными, а что константами?
- Какая функуция в итоге? Как записать то, что мы делали с начала и до конца?

In [ ]:
# Тут надо написать производную

grad = ...
grad

In [ ]:
alpha = 0.1
new_W = W - alpha * grad

new_W

In [ ]:
def count_grad(...):
    gard = ...
    return grad

Соберем все в одну функцию

In [ ]:
def regression_step(W, X, y, eps=1e-15, alpha=0.1):
    pred = predict(W, X, eps=1e-15, threshold=None)
    clip_pred = np.clip(pred, eps, 1 - eps)
    loss = compute_loss(clip_pred, y, eps=eps)
    grad = count_grad(...)
    new_W = W - alpha * grad
    return loss, new_W, np.linalg.norm(grad)

И теперь попробуем написать цикл обучения:
1. Делаем шаг (функция выше)
2. Проверяем, что норма градиента выше эпсилон (зачем?) и что лосс падает
3. Если в пункте 2 все ок, то повторяем 1-2

In [ ]:
def train_logreg(X, y, n_steps=100, eps = 1e-15, alpha=0.1):
    W = np.random.rand(X.shape[1])
    losses, norms, weights = [], [], []

    for i in range(n_steps):
        weights.append(W)

        ...

        losses.append(loss), norms.append(norm)

        if ...:
            print('Early stop')
            break

    return W, losses, norms, weights

In [ ]:
n_steps = 100
eps = 1e-15

W, losses, norms, w_hist = train_logreg(X, y, n_steps=100, eps = 1e-15)

Посмотрим, как меняются лоссы и нормы градиента в процессе обучения

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ax.plot(np.arange(n_steps), losses, label='losses')
ax.plot(np.arange(n_steps), norms, label='grad norms')
ax.legend();

Посмотрим, как мы учимся

In [ ]:
w_hist = np.array(w_hist)

In [ ]:
# Для красивой сетки значений

eps = 0.1
w1_vals = np.linspace(w_hist[:, 0].min() - eps, w_hist[:, 0].max() + eps, 20)
w2_vals = np.linspace(w_hist[:, 1].min() - eps, w_hist[:, 1].max() + eps, 20)
w1_grid, w2_grid = np.meshgrid(w1_vals, w2_vals)
loss_grid = np.zeros_like(w1_grid)

for i in range(w1_grid.shape[0]):
    for j in range(w1_grid.shape[1]):
        w = np.array([w1_grid[i, j], w2_grid[i, j]])
        pred = predict(w, X, eps=1e-15, threshold=None)
        loss_grid[i, j] = np.mean(compute_loss(pred, y))

In [ ]:
plt.figure(figsize=(8, 6))
contours = plt.contourf(w1_grid, w2_grid, loss_grid, levels=30, cmap='RdYlBu_r')
plt.colorbar(contours, label='Loss')

plt.scatter(w_hist[:, 0], w_hist[:, 1], label='Путь GD', color='black', s=1)
plt.scatter(w_hist[0, 0], w_hist[0, 1], c='green', label='Старт')
plt.scatter(w_hist[-1, 0], w_hist[-1, 1], c='red', label='Финиш')

plt.xlabel('w₁'), plt.ylabel('w₂')
plt.title('Путь градиентного спуска')
plt.legend()
plt.show()

In [ ]:
W

И посмотрим на качество

In [ ]:
preds = predict(W, X, eps=1e-15, threshold=0.5)
print(classification_report(y, preds))

Попробуем на настоящем датасете

In [ ]:
data = load_iris()
X, y, names = data['data'], data['target'], data['target_names']

Мы умеем решать только бинарную классификацию, так что один класс уберем

In [ ]:
idx = (data['target'] != 2)
X, y, names = X[idx], y[idx], names[:-1]

In [ ]:
n_steps = 100
eps = 1e-15

W, losses, norms, _ = train_logreg(X, y, n_steps=n_steps, eps=eps)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))

ax.plot(np.arange(len(losses)), losses, label='losses')
ax.plot(np.arange(len(norms)), norms, label='grad norms')
ax.legend();

In [ ]:
preds = predict(W, X, eps=1e-15, threshold=0.5)
print(classification_report(y, preds, target_names=names))